# Domain Adaptation Segmentation - One Kaggle Experiment

Runs one YOLO11s segmentation experiment on Kaggle, with durable logs and resume support. Default experiment: E01 source RGB baseline.

## 1. Settings

Attach the Kaggle dataset that contains `domain-adaptation-segmentation-kaggle.zip`. Keep internet enabled if packages need to be installed.

In [ ]:
from pathlib import Path

ZIP_PATH = None  # Example: "/kaggle/input/domain-adaptation-segmentation/domain-adaptation-segmentation-kaggle.zip"
REPO_DIR = Path("/kaggle/working/domain-adaptation-segmentation")
OUTPUT_ROOT = Path("/kaggle/working/runs/kaggle_single_e01")
REPORT_DIR = Path("reports/tables/kaggle_single_e01")

EXPERIMENT_CONFIG = "configs/experiments/e01_source_rgb_yolo11s.yaml"
YOLO_DEVICE = "0"      # Use "0" first for reliability on T4. Try "0,1" only after this works.
YOLO_BATCH = "8"       # T4-safe default. Try 16 if memory is stable.
YOLO_EPOCHS = "100"
YOLO_WORKERS = "2"
YOLO_PATIENCE = "25"
YOLO_RESUME = "auto"


## 2. Extract Project Package

In [ ]:
import zipfile

if ZIP_PATH is None:
    candidates = sorted(Path("/kaggle/input").glob("**/domain-adaptation-segmentation-kaggle.zip"))
    if not candidates:
        raise FileNotFoundError("Could not find domain-adaptation-segmentation-kaggle.zip under /kaggle/input")
    ZIP_PATH = str(candidates[0])

print("ZIP:", ZIP_PATH)
if not (REPO_DIR / "src" / "domain_adaptation_segmentation").exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(REPO_DIR)
else:
    print("Repository already extracted:", REPO_DIR)

print("Repo:", REPO_DIR)
print("Dataset YAML exists:", (REPO_DIR / "data/manifests/dataset_yamls/source_rgb.yaml").exists())


## 3. Install And Check GPU

In [ ]:
%cd /kaggle/working/domain-adaptation-segmentation
!pip install -q -r requirements.txt

import os
import torch

os.environ["PYTHONPATH"] = str(REPO_DIR / "src")
os.environ["OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ["REPORT_DIR"] = str(REPORT_DIR)
os.environ["EXPERIMENT_CONFIG"] = EXPERIMENT_CONFIG
os.environ["YOLO_DEVICE"] = YOLO_DEVICE
os.environ["YOLO_BATCH"] = YOLO_BATCH
os.environ["YOLO_EPOCHS"] = YOLO_EPOCHS
os.environ["YOLO_WORKERS"] = YOLO_WORKERS
os.environ["YOLO_PATIENCE"] = YOLO_PATIENCE
os.environ["YOLO_RESUME"] = YOLO_RESUME

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


## 4. Run One Experiment

This writes `status.json`, `stdout.log`, `stderr.log`, `results.csv`, and weights under `/kaggle/working/runs/kaggle_single_e01`.

In [ ]:
!bash scripts/remote/kaggle_run_one_experiment.sh


## 5. Inspect Results

In [ ]:
import json
import pandas as pd

run_dir = OUTPUT_ROOT / "experiments" / "E01_source_rgb_yolo11s"
status_path = run_dir / "status.json"
results_path = run_dir / "results.csv"
summary_path = REPO_DIR / REPORT_DIR / "summary_results.csv"

print("run_dir:", run_dir)
print("status exists:", status_path.exists())
if status_path.exists():
    print(json.dumps(json.loads(status_path.read_text()), indent=2)[:4000])

if results_path.exists():
    display(pd.read_csv(results_path).tail())

if summary_path.exists():
    display(pd.read_csv(summary_path))


## 6. Package Results For Download

In [ ]:
import shutil

bundle_dir = Path("/kaggle/working/kaggle_single_e01_artifacts")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)

if Path("/kaggle/working/runs").exists():
    shutil.copytree("/kaggle/working/runs", bundle_dir / "runs")
if (REPO_DIR / "reports").exists():
    shutil.copytree(REPO_DIR / "reports", bundle_dir / "reports")

archive_base = Path("/kaggle/working/kaggle_single_e01_results")
archive_path = shutil.make_archive(str(archive_base), "zip", bundle_dir)
print("Created:", archive_path)
